# Справочники КУАП и ГК → DRP

Читает файлы из той же папки, что Excel-отчёты, и заливает:

- `sbx_da.acq_kuap_inn` — все ИНН из файла КУАП
- `sbx_da.acq_gk_inn` — все ИНН из файла ГК (`is_exclude_gk = 1`)

В обоих файлах уже **только нужные ИНН**. Проверка «эта ГК из списка пяти?» не нужна: каждый ИНН из файла ГК исключается.

Инструкция: `HOW_TO_exclusions_gk.md`.

## Перед запуском
`/home/jovyan/documents/Equaring/Data/inn_list_kuap.txt`  
`/home/jovyan/documents/Equaring/Data/gk_list_inn.txt` (или `gk_inn_list.txt`)

Оба — обычный текст, **один ИНН на строку**. Не открывать через `read_csv` с авторазделителем.


In [ ]:
import getpass
import re
from pathlib import Path

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')

KUAP_CANDIDATES = [
    DATA_DIR / 'inn_list_kuap.txt',
    DATA_DIR / 'in_listkuap.txt',
    DATA_DIR / 'inn_list_kuap.csv',
]
GK_CANDIDATES = [
    DATA_DIR / 'gk_list_inn.txt',
    DATA_DIR / 'gk_inn_list.txt',
    DATA_DIR / 'gk_listinn.txt',
]

drp_schema = 'sbx_da'
kuap_table = 'acq_kuap_inn'
gk_table = 'acq_gk_inn'
drp_superset_grant_role = 'raisa_superset'

print('DATA_DIR', DATA_DIR, 'exists=', DATA_DIR.exists())
if DATA_DIR.exists():
    print('files in DATA_DIR:')
    for p in sorted(DATA_DIR.iterdir()):
        name = p.name.lower()
        if any(k in name for k in ('kuap', 'gk', 'инн', 'inn')):
            print(' ', p.name, p.stat().st_size, 'bytes')


In [ ]:
def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None


kuap_path = first_existing(KUAP_CANDIDATES)
gk_path = first_existing(GK_CANDIDATES)
print('KUAP file:', kuap_path)
print('GK file:  ', gk_path)
if kuap_path is None or gk_path is None:
    raise FileNotFoundError(
        'Не найден inn_list_kuap.txt или gk_list_inn.txt в DATA_DIR. '
        f'DATA_DIR={DATA_DIR}'
    )


In [ ]:
# КУАП: обычный txt, один ИНН на строку.
# Не используем read_csv(sep=None) — Sniffer принимает частую цифру (у вас «7») за разделитель
# и режет 77331725760 → 3311 | 25760.

def read_text_lines(path):
    raw_bytes = path.read_bytes()
    text = None
    used_enc = None
    for enc in ('utf-8-sig', 'utf-8', 'cp1251'):
        try:
            text = raw_bytes.decode(enc)
            used_enc = enc
            break
        except UnicodeDecodeError:
            continue
    if text is None:
        text = raw_bytes.decode('utf-8', errors='replace')
        used_enc = 'utf-8-replace'
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    lines = [ln.strip().replace('\ufeff', '') for ln in text.split('\n')]
    return used_enc, [ln for ln in lines if ln]


def inns_from_txt_line(line):
    s = str(line).strip()
    if re.fullmatch(r'(?i)инн|inn', s):
        return []
    s = re.sub(r'(?i)^(инн|inn)\s*[:\-]?\s*', '', s).strip()
    parts = re.split(r'[\s,;]+', s) if re.search(r'[\s,;]', s) else [s]
    return [p for p in parts if p]


used_enc, kuap_lines = read_text_lines(kuap_path)
print('KUAP encoding:', used_enc, '| path:', kuap_path)
print('KUAP raw lines:', len(kuap_lines))
display(pd.DataFrame({'raw_line': kuap_lines}))

kuap_vals = []
skipped = []
for line in kuap_lines:
    extracted = inns_from_txt_line(line)
    if not extracted:
        skipped.append({'raw_line': line, 'reason': 'заголовок/пусто'})
        continue
    ok = False
    for cell in extracted:
        inn = normalize_inn_q1(cell)
        if inn:
            kuap_vals.append(inn)
            ok = True
        else:
            skipped.append({'raw_line': line, 'reason': f'не ИНН 10/12: {cell!r}'})
    if not ok and extracted:
        pass

kuap_df = pd.DataFrame({'inn': kuap_vals}).drop_duplicates().sort_values('inn').reset_index(drop=True)
print('KUAP unique INN =', len(kuap_df))
if skipped:
    print('Пропущенные строки:')
    display(pd.DataFrame(skipped))
if kuap_df.empty:
    raise RuntimeError('Список КУАП пуст после нормализации ИНН')
display(kuap_df)


In [ ]:
# ГК: тот же txt, что КУАП — один ИНН на строку. Все строки исключаются.
# Не read_csv(sep=None) и не Excel.

if gk_path.suffix.lower() in ('.xlsx', '.xls'):
    raise RuntimeError(
        f'Ожидается txt, найден Excel: {gk_path.name}. '
        'Положите gk_list_inn.txt (один ИНН на строку) в DATA_DIR.'
    )

used_enc_gk, gk_lines = read_text_lines(gk_path)
print('GK encoding:', used_enc_gk, '| path:', gk_path)
print('GK raw lines:', len(gk_lines))
display(pd.DataFrame({'raw_line': gk_lines}))

gk_vals = []
gk_skipped = []
for line in gk_lines:
    extracted = inns_from_txt_line(line)
    if not extracted:
        gk_skipped.append({'raw_line': line, 'reason': 'заголовок/пусто'})
        continue
    for cell in extracted:
        inn = normalize_inn_q1(cell)
        if inn:
            gk_vals.append(inn)
        else:
            gk_skipped.append({'raw_line': line, 'reason': f'не ИНН 10/12: {cell!r}'})

gk_df = pd.DataFrame({'inn': gk_vals}).drop_duplicates().sort_values('inn').reset_index(drop=True)
gk_df['gk_name'] = 'ГК'
gk_df['is_exclude_gk'] = '1'

print('GK unique INN =', len(gk_df))
if gk_skipped:
    print('Пропущенные строки:')
    display(pd.DataFrame(gk_skipped))
if gk_df.empty:
    raise RuntimeError('Список ГК пуст после нормализации ИНН')
display(gk_df)


In [ ]:
drp_user = input('DRP user: ').strip()
drp_password = getpass.getpass('DRP password: ')

drp = connect(
    to='DRP',
    user_params={'user_name': drp_user, 'password': drp_password},
)
print('DRP connected as', drp_user)


In [ ]:
def upload_text_table(fq, df, grant=True):
    upload_df = df.copy()
    for c in upload_df.columns:
        upload_df[c] = upload_df[c].map(lambda x: None if pd.isna(x) else str(x)).astype(object)
    col_defs = [f'"{str(c)}" TEXT' for c in upload_df.columns]
    create_sql = f'CREATE TABLE {fq} (\n  ' + ',\n  '.join(col_defs) + '\n)'
    with drp:
        drp.execute(f'DROP TABLE IF EXISTS {fq}')
        drp.execute(create_sql)
        drp.write(table=fq, df=upload_df, mode='append')
        cnt = drp.fetch(f'SELECT COUNT(*) AS row_cnt FROM {fq}')
        if grant:
            try:
                drp.execute(f'GRANT USAGE ON SCHEMA {drp_schema} TO {drp_superset_grant_role}')
                drp.execute(f'GRANT SELECT ON TABLE {fq} TO {drp_superset_grant_role}')
                print(f'OK: GRANT SELECT ON {fq} TO {drp_superset_grant_role}')
            except Exception as grant_exc:
                print('WARNING: GRANT failed:', type(grant_exc).__name__, str(grant_exc)[:400])
                print(f'  GRANT SELECT ON TABLE {fq} TO {drp_superset_grant_role};')
    n = int(pd.to_numeric(cnt.iloc[0, 0], errors='coerce'))
    print(f'OK {fq} rows =', n)
    return n


kuap_fq = f'{drp_schema}.{kuap_table}'
gk_fq = f'{drp_schema}.{gk_table}'
upload_text_table(kuap_fq, kuap_df)
upload_text_table(gk_fq, gk_df[['inn', 'gk_name', 'is_exclude_gk']])


In [ ]:
# Smoke: все ИНН из обоих файлов — исключения
smoke_sql = f'''
SELECT
  (SELECT COUNT(*) FROM {kuap_fq}) AS kuap_inns,
  (SELECT COUNT(*) FROM {gk_fq}) AS gk_inns_all,
  (SELECT COUNT(*) FROM {gk_fq} WHERE BTRIM(is_exclude_gk) = '1') AS gk_inns_exclude,
  (
    SELECT COUNT(DISTINCT NULLIF(BTRIM(CAST(d.inn AS TEXT)), ''))
    FROM sbx_da.tmp_shestopalov_acq_datamart_final_script_2 d
    JOIN {kuap_fq} k
      ON k.inn = NULLIF(BTRIM(CAST(d.inn AS TEXT)), '')
  ) AS datamart_kuap_inns,
  (
    SELECT COUNT(DISTINCT NULLIF(BTRIM(CAST(d.agr_id AS TEXT)), ''))
    FROM sbx_da.tmp_shestopalov_acq_datamart_final_script_2 d
    JOIN {kuap_fq} k
      ON k.inn = NULLIF(BTRIM(CAST(d.inn AS TEXT)), '')
  ) AS datamart_kuap_agr,
  (
    SELECT COUNT(DISTINCT NULLIF(BTRIM(CAST(d.inn AS TEXT)), ''))
    FROM sbx_da.tmp_shestopalov_acq_datamart_final_script_2 d
    JOIN {gk_fq} g
      ON g.inn = NULLIF(BTRIM(CAST(d.inn AS TEXT)), '')
     AND BTRIM(CAST(g.is_exclude_gk AS TEXT)) = '1'
  ) AS datamart_gk_excl_inns,
  (
    SELECT COUNT(DISTINCT NULLIF(BTRIM(CAST(d.agr_id AS TEXT)), ''))
    FROM sbx_da.tmp_shestopalov_acq_datamart_final_script_2 d
    JOIN {gk_fq} g
      ON g.inn = NULLIF(BTRIM(CAST(d.inn AS TEXT)), '')
     AND BTRIM(CAST(g.is_exclude_gk AS TEXT)) = '1'
  ) AS datamart_gk_excl_agr
'''
with drp:
    smoke = drp.fetch(smoke_sql)
    gk_preview = drp.fetch(
        f'''
        SELECT gk_name, is_exclude_gk, COUNT(*) AS inns
        FROM {gk_fq}
        WHERE BTRIM(is_exclude_gk) = '1'
        GROUP BY gk_name, is_exclude_gk
        ORDER BY inns DESC
        '''
    )
display(smoke)
display(gk_preview)

print('Next:')
print('1. SQL Lab: блок 2d / vd_acq_tsp_efficiency_period_cols_excl.sql')
print('2. Чарт v2_period_filial_tsp_efficiency_cols_excl')
print('3. HOW_TO_exclusions_gk.md')
